# P4 Depth Completion — Research-Driven Kaggle Experiments

**Branch:** `feat/research-driven-depth-completion`
**Run this notebook top-to-bottom on Kaggle with a GPU runtime.**

## Experiment map
| ID | Cell | Description |
|---|---|---|
| EXP-001 | 5 | Interpolation baselines (nearest/bilinear/bicubic), x4 clean n=100 |
| EXP-002 | 6 | Improved classical (classical_blend), x4 clean n=100 |
| EXP-003 | 7 | Deep branch (SGNet x4), x4 clean n=100 — needs checkpoint |
| EXP-004 | 8 | Improved hybrid v2 (regime-aware + hole-band), x4 clean n=100 |
| EXP-005 | 9 | Hybrid fusion ablation (fixed-alpha sweep), x4 clean n=50 |
| EXP-006 | 10 | Robustness: holes + noise, x4 n=50 each |
| EXP-007 | 11 | Full matrix x2/x4/x8, n=100 |
| EXP-D   | 13 | Official hackathon testset, native missing depth, x2/x4/x8 |

**Honesty protocol:** quote as "Kaggle run, <date>, <GPU>, feat/research-driven-depth-completion".
Missing checkpoints → deep/hybrid reported *not evaluated*, never silently scored.
Metrics: RMSE / MAE / absRel / delta1 / iRMSE / iMAE / RMSE-hole / PSNR / SSIM / boundary-RMSE / edge-F1 / latency.


In [ ]:
#@title Cell 1 — Environment
import sys, platform, torch
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.cuda.synchronize()
    p = torch.cuda.get_device_properties(0)
    print(f"gpu: {p.name}  vram: {p.total_memory/2**30:.1f} GiB")
else:
    print("gpu: none — CPU mode")


In [ ]:
#@title Cell 2 — Clone repo and install dependencies
import os, subprocess, sys

REPO   = "/kaggle/working/CV-Hackthon"
BRANCH = "feat/research-driven-depth-completion"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/KumarSrinidhi/CV-Hackthon.git", REPO],
                   check=True)
os.chdir(REPO)

reqs = [l.strip() for l in open("requirements.txt")
        if l.strip() and not l.startswith("#")]
reqs = [r for r in reqs if not r.lower().startswith(("torch", "torchvision"))]
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *reqs, "h5py", "huggingface_hub"], check=True)
sys.path.insert(0, REPO)

branch = subprocess.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"repo: {os.getcwd()}")
print(f"branch: {branch}  commit: {commit}")


## Checkpoints (required for EXP-003/004/005/007)

Attach a Kaggle dataset containing `SGNet_X4_R.pth` / `SGNet_X8_R.pth` / `SGNet_X16_R.pth`
**directly in the folder root** (not in a subfolder), then set `CKPT_DIR` below.
Without checkpoints, `deep`/`hybrid*` are reported as *not evaluated* — never silently scored.


In [ ]:
#@title Cell 3 — Checkpoint status
CKPT_DIR = ""  #@param {type:"string"}

import os
if CKPT_DIR.strip():
    os.environ["SGNET_CKPT_DIR"] = CKPT_DIR.strip()

from deep.settings import ckpt_dir
from deep.deep_model import resolve_checkpoint

print("checkpoint dir:", ckpt_dir())
for s in (2, 4, 8, 16):
    try:
        p = resolve_checkpoint(s)
        print(f"  x{s}: {p.name}  ({p.stat().st_size/2**20:.0f} MB)")
    except Exception as e:
        print(f"  x{s}: NOT AVAILABLE ({type(e).__name__})")


## Data — HF NYU val tar (~1 GB)

With Internet ON the next cell downloads from HuggingFace automatically.
Alternatively attach a Kaggle dataset containing `val-000000.tar` and set `TAR_PATH`.


In [ ]:
#@title Cell 4 — Locate or download HF val tar
TAR_PATH = ""  #@param {type:"string"}

import glob, os

def _find_attached():
    hits = glob.glob("/kaggle/input/**/val-*.tar", recursive=True)
    return hits[0] if hits else ""

if TAR_PATH.strip():
    TAR = TAR_PATH.strip()
elif _find_attached():
    TAR = _find_attached()
else:
    from huggingface_hub import HfApi, hf_hub_download
    files = HfApi().list_repo_files("sayakpaul/nyu_depth_v2", repo_type="dataset")
    cands = [f for f in files if f.endswith(".tar")]
    val_cands = [f for f in cands if "val" in os.path.basename(f).lower()] or cands
    TAR = hf_hub_download("sayakpaul/nyu_depth_v2", repo_type="dataset",
                          filename=sorted(val_cands)[0])
print(f"tar: {TAR}  ({os.path.getsize(TAR)/1e6:.0f} MB)")


## EXP-001 — Interpolation baselines

Nearest / bilinear / bicubic at x4 clean, n=100.
These are the reference numbers every other experiment is compared against.
**Seed:** 0 | **Dataset:** HF NYU val | **Scale:** x4 | **Degradation:** clean bicubic


In [ ]:
#@title EXP-001 — baselines x4 clean n=100
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["nearest", "bilinear", "bicubic"], max_samples=100)
save_results(res, "results/kaggle_exp001_baselines", "hf_nyu_val_x4_baselines")
print(summary_markdown(res))


## EXP-002 — Improved classical (classical_blend)

`classical_hr_depth()` now uses `classical_blend` by default:
iterative bicubic back-projection blended 40% with the edge-sharpened dense path.
Prior validation on NYU val150 clean: RMSE **0.0250** vs bicubic 0.0292 (+14%).
**Seed:** 0 | **Scale:** x4 | **n:** 100 | **Degradation:** clean


In [ ]:
#@title EXP-002 — improved classical vs bicubic, x4 clean n=100
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "classical"], max_samples=100)
save_results(res, "results/kaggle_exp002_classical", "hf_nyu_val_x4_classical")
print(summary_markdown(res))


## EXP-003 — Deep branch (SGNet x4)

Requires `SGNet_X4_R.pth`. Without it this cell reports *not evaluated*.
Regime-aware adaptive checkpoint selection is active (clean vs robust curriculum).
**Seed:** 0 | **Scale:** x4 | **n:** 100 | **Degradation:** clean


In [ ]:
#@title EXP-003 — SGNet deep branch, x4 clean n=100
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "deep"], max_samples=100)
save_results(res, "results/kaggle_exp003_deep", "hf_nyu_val_x4_deep")
print(summary_markdown(res))


## EXP-004 — Improved hybrid v2

Regime-aware fusion with hole-band classical confidence boost.
- Clean/noise regime: classical weight = 0.0 (deep dominates)
- Holes regime: classical weight = 0.05 (validated)
- Hole-band: classical confidence raised to 0.5 around missing LR pixels
- Measurement consistency applied on clean/noise (disabled on noise to avoid anchoring sensor noise)
**Seed:** 0 | **Scale:** x4 | **n:** 100 | **Degradation:** clean


In [ ]:
#@title EXP-004 — improved hybrid v2, x4 clean n=100
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds  = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
res = evaluate_methods(ds, ["bicubic", "deep", "hybrid"], max_samples=100)
save_results(res, "results/kaggle_exp004_hybrid", "hf_nyu_val_x4_hybrid")
print(summary_markdown(res))


## EXP-005 — Hybrid fusion ablation (fixed-alpha sweep)

Sweep classical weight alpha in {0.0, 0.02, 0.05, 0.10, 0.20} with and without
measurement consistency. Prior validation (NYU val30): alpha=0.02 gave lowest RMSE.
**Seed:** 0 | **Scale:** x4 | **n:** 50 | **Degradation:** clean


In [ ]:
#@title EXP-005 — fixed-alpha ablation sweep, x4 clean n=50
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

ds = HFNYUValDataset(4, DegradationConfig(), tar_paths=[TAR])
methods = [
    "deep",
    "fixed_0.0", "fixed_0.02", "fixed_0.05", "fixed_0.10", "fixed_0.20",
    "fixed_consistency_0.0", "fixed_consistency_0.02", "fixed_consistency_0.05",
]
res = evaluate_methods(ds, methods, max_samples=50)
save_results(res, "results/kaggle_exp005_ablation", "hf_nyu_val_x4_ablation")
print(summary_markdown(res))


## EXP-006 — Robustness: holes and noise

- **Holes:** 10% of LR pixels randomly dropped (`hole_ratio=0.10`)
- **Noise:** Gaussian noise std=0.02 m (`noise_std=0.02`)

Prior finding (master branch, Tesla T4): deep < bicubic under 10% holes.
The improved hybrid v2 with hole-band confidence should recover this.
**Seed:** 0 | **Scale:** x4 | **n:** 50 per regime


In [ ]:
#@title EXP-006 — robustness holes + noise, x4 n=50 each
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["bicubic", "classical", "deep", "hybrid"]

for tag, deg in (
    ("holes", DegradationConfig(hole_ratio=0.10)),
    ("noise", DegradationConfig(noise_std=0.02)),
):
    ds  = HFNYUValDataset(4, deg, tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=50)
    save_results(res, f"results/kaggle_exp006_{tag}", f"hf_nyu_val_x4_{tag}")
    print(f"--- {tag} ---")
    print(summary_markdown(res))


## EXP-007 — Full comparison matrix x2/x4/x8

All methods across all supported scales, n=100 per scale.
Reports all metrics: RMSE / MAE / absRel / delta1 / iRMSE / iMAE / RMSE-hole / PSNR / SSIM / boundary-RMSE / edge-F1 / latency.


In [ ]:
#@title EXP-007 — full matrix x2/x4/x8, n=100
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

METHODS = ["nearest", "bilinear", "bicubic", "classical",
           "deep", "hybrid", "hybrid_fixed02", "hybrid_adaptive"]

for scale in (2, 4, 8):
    ds  = HFNYUValDataset(scale, DegradationConfig(), tar_paths=[TAR])
    res = evaluate_methods(ds, METHODS, max_samples=100)
    save_results(res, f"results/kaggle_exp007_x{scale}", f"hf_nyu_val_x{scale}_final")
    print(f"=== x{scale} ===")
    print(summary_markdown(res))


## EXP-D — Official hackathon testset (20 samples, native missing depth)

`data/depthsr_testset/` from branch `kumarbro`: 20 synthetic samples with block-mean
LR depth and sensor-like holes. Methods receive the native LR depth + validity mask
(not re-degraded). Hybrid runs with `use_consistency=False` (native operator is
block-mean, not bicubic).


In [ ]:
#@title EXP-D step 1 — extract official testset from kumarbro branch
import subprocess
subprocess.run(["git", "fetch", "-q", "origin", "kumarbro"], check=True)
subprocess.run(["git", "checkout", "-q", "FETCH_HEAD", "--", "data/depthsr_testset"],
               check=True)
import os
samples = sorted(d for d in os.listdir("data/depthsr_testset")
                 if os.path.isdir(os.path.join("data/depthsr_testset", d)))
print(f"{len(samples)} samples:", samples)


In [ ]:
#@title EXP-D step 2 — evaluate testset x2/x4/x8
import os
import numpy as np
from PIL import Image
from data.dataset import DepthSRDataset
from data.preprocessing import to_float_rgb
from eval.evaluate import evaluate_methods, save_results, summary_markdown
from hybrid.pipeline import hybrid_hr_depth
from deep.deep_model import resolve_checkpoint


class DepthSRTestset(DepthSRDataset):
    name = "depthsr_testset"
    unit = "m"
    data_range = 10.0

    def __init__(self, scale, root="data/depthsr_testset"):
        super().__init__(scale)
        self.folders = sorted(
            os.path.join(root, d) for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )

    def _num_raw(self):
        return len(self.folders)

    def _load_raw(self, i):
        d = self.folders[i]
        rgb = np.asarray(Image.open(os.path.join(d, "rgb.png")))
        gt  = np.load(os.path.join(d, "depth_hr.npy")).astype(np.float32)
        return os.path.basename(d), to_float_rgb(rgb), gt, np.isfinite(gt) & (gt > 0)

    def __getitem__(self, i):
        name, rgb, gt, valid = self._load_raw(i)
        d    = self.folders[i]
        lr   = np.load(os.path.join(d, f"lr_depth_{self.scale}x.npy")).astype(np.float32)
        mask = np.load(os.path.join(d, f"lr_mask_{self.scale}x.npy")).astype(bool)
        return {"name": name,
                "hr_rgb": np.ascontiguousarray(rgb, np.float32),
                "depth_original": gt, "valid_mask_original": valid,
                "lr_depth": lr, "lr_valid_mask": mask, "lr_depth_clean": lr.copy(),
                "scale": self.scale, "unit": self.unit,
                "data_range": float(self.data_range), "operator_exact": False}


def _hybrid_testset(lr, rgb, m, s=4):
    try:
        resolve_checkpoint(s)
    except Exception as e:
        raise NotImplementedError(f"deep checkpoint unavailable: {e}") from e
    return hybrid_hr_depth(lr, rgb, m, s, use_consistency=False)


testset_methods = {"hybrid": _hybrid_testset}

for s in (2, 4, 8):
    ds  = DepthSRTestset(s)
    res = evaluate_methods(
        ds,
        ["nearest", "bilinear", "bicubic", "classical", "deep",
         "hybrid", "hybrid_fixed02", "hybrid_adaptive"],
        methods=testset_methods,
    )
    save_results(res, f"results/kaggle_testset_x{s}", f"testset_x{s}")
    print(f"=== testset x{s} ===")
    print(summary_markdown(res))


## Results

Every experiment writes to `/kaggle/working/CV-Hackthon/results/kaggle_*/`:
- `*_summary.md` — printed table (lists *not evaluated* methods with reasons)
- `*_summary.json` — machine-readable summary + metadata
- `*_per_image.csv` — per-sample metrics + latency

Download via Kaggle **Download output** (right sidebar) or use the dump cell below.


In [ ]:
#@title Cell — Print all summary tables from this run
from IPython.display import Markdown, display
import glob

for md_path in sorted(glob.glob("results/kaggle_*/*summary.md")):
    print(f"--- {md_path} ---")
    display(Markdown(open(md_path).read()))


## FINAL DUMP — All results in one self-contained block

Run this cell last. It prints a single fenced JSON block containing every
experiment's summary, metadata, and per-image rows. The output is stored in
the notebook's cell output so you can read it directly from the `.ipynb` file
or paste it back for analysis.


In [ ]:
#@title FINAL DUMP — all results as one JSON block (read directly from .ipynb)
import glob, json, csv, os
from pathlib import Path

# ── collect all result JSON files ────────────────────────────────────────────
all_results = {}

for json_path in sorted(glob.glob("results/kaggle_*/*summary.json")):
    exp_tag = Path(json_path).parent.name
    data    = json.loads(open(json_path).read())
    all_results[exp_tag] = data

# ── collect per-image CSVs ───────────────────────────────────────────────────
per_image = {}
for csv_path in sorted(glob.glob("results/kaggle_*/*per_image.csv")):
    exp_tag = Path(csv_path).parent.name
    with open(csv_path, newline="") as f:
        per_image[exp_tag] = list(csv.DictReader(f))

# ── collect run environment ──────────────────────────────────────────────────
import sys, platform, subprocess, torch

env = {
    "python":    sys.version.split()[0],
    "torch":     torch.__version__,
    "cuda":      torch.cuda.is_available(),
    "gpu":       torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "platform":  platform.platform(),
}
try:
    env["branch"] = subprocess.check_output(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
    env["commit"] = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"]).decode().strip()
except Exception:
    pass

# ── build the dump ───────────────────────────────────────────────────────────
dump = {
    "run_env":     env,
    "experiments": all_results,
    "per_image":   per_image,
}

# ── print as a single fenced block so it is readable in the .ipynb ───────────
print("```json")
print(json.dumps(dump, indent=2))
print("```")

# ── also save to a single file for easy download ─────────────────────────────
out_path = "results/kaggle_ALL_RESULTS_DUMP.json"
Path(out_path).write_text(json.dumps(dump, indent=2))
print(f"\nSaved consolidated dump to: {out_path}")
print(f"Experiments: {list(all_results.keys())}")
print(f"Total per-image rows: {sum(len(v) for v in per_image.values())}")
